# Day 10 — Async Patterns: gather, wait_for, create_task

=======================================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - asyncio.gather()     — run multiple coroutines IN PARALLEL
  - asyncio.wait_for()   — add a timeout to any coroutine
  - asyncio.create_task() — fire-and-forget background writes

WHY THIS MATTERS FOR LLM ENGINEERING:
  An LLM agent often needs to call multiple tools before answering.
  Sequential:  3 tools × 2 seconds each = 6 seconds
  Parallel:    3 tools simultaneously   = 2 seconds
  asyncio.gather() gives you that 3x speedup for free.


In [ ]:

import asyncio
import logging
import time
from pathlib import Path

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")




## SIMULATED TOOL FUNCTIONS

(In Module 05 these become real MCP tools called by LangGraph agents)
═════════════════════════════════════════════════════════════════════════════


In [ ]:

async def tool_lookup_customer(customer_id: int) -> dict:
    """Simulate fetching a customer from the database."""
    log.info(f"→ lookup_customer({customer_id}) started")
    await asyncio.sleep(1.2)   # simulates DB query latency
    log.info(f"← lookup_customer({customer_id}) done")
    return {
        "customer_id": customer_id,
        "name"       : "Danielle Johnson",
        "email"      : "john21@example.net",
        "city"       : "Port Matthew",
    }



In [ ]:
async def tool_lookup_order(order_id: int) -> dict:
    """Simulate fetching an order from the database."""
    log.info(f"→ lookup_order({order_id}) started")
    await asyncio.sleep(1.5)   # orders query is slightly slower
    log.info(f"← lookup_order({order_id}) done")
    return {
        "order_id"    : order_id,
        "status"      : "In Transit",
        "total_amount": 1324.89,
        "product"     : "Classic Monitor",
    }



In [ ]:
async def tool_check_inventory(product_id: int) -> dict:
    """Simulate checking inventory from the warehouse system."""
    log.info(f"→ check_inventory({product_id}) started")
    await asyncio.sleep(0.8)   # inventory is cached, faster
    log.info(f"← check_inventory({product_id}) done")
    return {
        "product_id"      : product_id,
        "quantity_available": 238,
        "warehouse"       : "East Coast Distribution Center",
    }



In [ ]:
async def tool_slow_shipping_api(order_id: int) -> dict:
    """Simulate a slow external shipping API call (will timeout)."""
    log.info(f"→ slow_shipping_api({order_id}) started (will take 8s...)")
    await asyncio.sleep(8.0)   # very slow — intentionally over the timeout
    return {"tracking": "1Z999AA133781932"}




## SECTION 1: asyncio.gather() — PARALLEL EXECUTION

═════════════════════════════════════════════════════════════════════════════


In [ ]:

async def run_tools_sequentially(
    customer_id: int,
    order_id   : int,
    product_id : int,
) -> tuple[dict, dict, dict]:
    """
    Run three tool calls ONE AFTER ANOTHER.

    Total time = 1.2s + 1.5s + 0.8s = ~3.5 seconds.
    Every second here is a second the user waits.
    """

    customer  = await tool_lookup_customer(customer_id)
    order     = await tool_lookup_order(order_id)
    inventory = await tool_check_inventory(product_id)

    return customer, order, inventory



In [ ]:
async def run_tools_in_parallel(
    customer_id: int,
    order_id   : int,
    product_id : int,
) -> tuple[dict, dict, dict]:
    """
    Run three tool calls SIMULTANEOUSLY using asyncio.gather().

    asyncio.gather(*coroutines) schedules all coroutines at once.
    The event loop runs them concurrently — as soon as one awaits,
    another one runs. The total time is max(1.2, 1.5, 0.8) = ~1.5s.

    Return order: gather() always returns results in the SAME ORDER
    as the coroutines were passed in, regardless of completion order.

    Args:
        customer_id: ID of customer to look up.
        order_id   : ID of order to look up.
        product_id : ID of product to check inventory for.

    Returns:
        Tuple of (customer_dict, order_dict, inventory_dict).
    """

    # All three coroutines start simultaneously
    # The event loop switches between them as each one awaits
    customer, order, inventory = await asyncio.gather(
        tool_lookup_customer(customer_id),
        tool_lookup_order(order_id),
        tool_check_inventory(product_id),
    )

    return customer, order, inventory



In [ ]:
async def run_tools_with_partial_failure(
    customer_id: int,
    order_id   : int,
    product_id : int,
) -> list[dict | Exception]:
    """
    Run tools in parallel but handle individual failures gracefully.

    Without return_exceptions=True:
        If ANY coroutine raises an exception, gather() immediately
        raises it and cancels remaining coroutines.

    With return_exceptions=True:
        gather() collects ALL results AND exceptions.
        Exceptions are returned as values in the results list.
        You can then handle each individually.

    This is the production pattern — in a real agent, one tool
    failing should not crash the entire response.

    Returns:
        List of results — each item is either a dict or an Exception.
    """

    # Simulate a broken tool by raising an exception
    async def broken_tool(product_id: int) -> dict:
        await asyncio.sleep(0.3)
        raise ConnectionError(f"Inventory service unavailable for product {product_id}")

    results = await asyncio.gather(
        tool_lookup_customer(customer_id),
        tool_lookup_order(order_id),
        broken_tool(product_id),          # this one will fail
        return_exceptions=True,           # collect errors instead of raising
    )

    # Process results — check each one
    processed = []
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            log.warning(f"Tool {i} failed: {result}")
            processed.append({"error": str(result)})
        else:
            processed.append(result)

    return processed




## SECTION 2: asyncio.wait_for() — TIMEOUTS

═════════════════════════════════════════════════════════════════════════════


In [ ]:

async def call_with_timeout(
    coroutine_fn,
    timeout_seconds: float = 3.0,
    fallback: dict | None = None,
) -> dict:
    """
    Run a coroutine with a timeout. Return fallback if it exceeds the limit.

    asyncio.wait_for(coro, timeout=N) cancels the coroutine after N seconds
    and raises asyncio.TimeoutError.

    In production: a stuck external API should NEVER block user responses.
    Always add a timeout to any external call.

    Args:
        coroutine_fn   : The coroutine to run (async def function call).
        timeout_seconds: Max seconds to wait before cancelling.
        fallback       : Dict to return if timeout occurs.

    Returns:
        The coroutine result, or the fallback dict if timeout occurs.
    """

    if fallback is None:
        fallback = {"error": "timeout", "message": "Service did not respond in time"}

    try:
        # asyncio.wait_for() adds a deadline to any coroutine
        result = await asyncio.wait_for(coroutine_fn, timeout=timeout_seconds)
        return result

    except asyncio.TimeoutError:
        log.warning(
            f"Tool call timed out after {timeout_seconds}s. "
            f"Using fallback response."
        )
        return fallback




## SECTION 3: asyncio.create_task() — FIRE AND FORGET

═════════════════════════════════════════════════════════════════════════════


In [ ]:

# Global task registry — keeps track of background tasks
# Without this, tasks can be garbage-collected before they finish!
_background_tasks: set[asyncio.Task] = set()



In [ ]:
async def write_llm_call_log(
    query   : str,
    response: str,
    latency_ms: float,
) -> None:
    """
    Write an LLM call record to the log (simulated).

    This is the SLOW part — writing to a database or cloud service.
    We don't want the user to wait for this before getting their response.
    So we fire it as a background task.

    Args:
        query     : The user query that was sent.
        response  : The LLM response received.
        latency_ms: How long the LLM call took.
    """

    log.info(f"[BG TASK] Writing log: latency={latency_ms}ms")
    await asyncio.sleep(0.5)   # simulate slow I/O (database write)
    log.info(f"[BG TASK] Log written successfully")



In [ ]:
async def handle_user_query_with_background_log(query: str) -> str:
    """
    Handle a user query AND log the call, without waiting for the log.

    Pattern:
    1. Call the LLM and get a response
    2. Fire the log write as a background task (don't await it)
    3. Return the response to the user immediately
    4. The background task finishes on its own

    This is the same pattern FastAPI's BackgroundTasks uses internally.

    Args:
        query: The user's message.

    Returns:
        The LLM response (returned immediately, log happens in background).
    """

    start = time.perf_counter()

    # Simulate the LLM call (the part the user WAITS for)
    await asyncio.sleep(0.5)
    response = f"[LLM] Answer to: '{query}'"

    latency_ms = round((time.perf_counter() - start) * 1000)

    # Create a background task for the log write
    # asyncio.create_task() SCHEDULES the coroutine but does NOT await it
    # The function returns immediately without waiting for the write to finish
    task = asyncio.create_task(
        write_llm_call_log(query, response, latency_ms)
    )

    # IMPORTANT: Store the task reference to prevent garbage collection!
    # If you don't store it, the task may be cancelled before it finishes.
    # The _background_tasks set keeps a reference alive.
    _background_tasks.add(task)

    # Remove the task from the set when it finishes (cleanup)
    task.add_done_callback(_background_tasks.discard)

    # Return the response NOW — the log write happens in the background
    log.info(f"Response returned. Background log task scheduled.")
    return response




## MAIN DEMO

═════════════════════════════════════════════════════════════════════════════


In [ ]:

async def main():
    print("=" * 60)
    print("DAY 10 — Async Patterns")
    print("=" * 60)

    # gather: sequential vs parallel
    print("\n[1] Sequential vs Parallel Tool Calls")
    CUSTOMER_ID, ORDER_ID, PRODUCT_ID = 1001, 3001, 2001

    start = time.perf_counter()
    seq_results = await run_tools_sequentially(CUSTOMER_ID, ORDER_ID, PRODUCT_ID)
    seq_time = round(time.perf_counter() - start, 2)

    start = time.perf_counter()
    par_results = await run_tools_in_parallel(CUSTOMER_ID, ORDER_ID, PRODUCT_ID)
    par_time = round(time.perf_counter() - start, 2)

    print(f"\n  Sequential: {seq_time}s")
    print(f"  Parallel  : {par_time}s")
    print(f"  Speedup   : {seq_time / par_time:.1f}x")
    print(f"  Customer  : {par_results[0]['name']}")
    print(f"  Order     : {par_results[1]['status']}")
    print(f"  Inventory : {par_results[2]['quantity_available']} units")

    # gather with partial failure
    print("\n[2] Partial Failure Handling (return_exceptions=True)")
    partial_results = await run_tools_with_partial_failure(CUSTOMER_ID, ORDER_ID, PRODUCT_ID)
    for i, result in enumerate(partial_results):
        status = "ERROR" if "error" in result else "OK"
        print(f"  Tool {i}: {status} — {result}")

    # wait_for: timeout
    print("\n[3] Timeout with Fallback")
    start = time.perf_counter()
    result = await call_with_timeout(
        tool_slow_shipping_api(ORDER_ID),
        timeout_seconds=2.0,
        fallback={"tracking": "unavailable", "message": "Tracking service slow, try again later"},
    )
    elapsed = round(time.perf_counter() - start, 2)
    print(f"  Result  : {result}")
    print(f"  Elapsed : {elapsed}s (timed out at 2s, not 8s)")

    # create_task: fire and forget
    print("\n[4] Fire and Forget Background Task")
    start = time.perf_counter()
    response = await handle_user_query_with_background_log("What is the status of order 3001?")
    response_time = round((time.perf_counter() - start) * 1000)
    print(f"  Response returned in {response_time}ms: {response}")
    print(f"  (Background log task still running...)")

    # Wait a moment to let the background task finish before the demo ends
    await asyncio.sleep(1.0)
    print(f"  (Background task finished)")


if __name__ == "__main__":
    asyncio.run(main())


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day10_async_patterns.py', run_name='__main__')
